# Air Pollution in NYC
**Authors:** Vincent Vocke & Abrar Daiyan Khan
**Course:** DS Studio II
**Dataset:** NYC Environment & Health Data Portal, Air Quality Surveillance (2005-2023)

This project looks at how air pollution moves across NYC neighborhoods over time, whether we can predict it, and who ends up bearing the most exposure. We focused on three pollutants: PM2.5 (fine particles), NO2 (nitrogen dioxide), and Ozone. The data covers 42 neighborhood health districts from 2009 to 2023.

| Part | What we do |
|------|------------|
| 1 | Load and clean the raw data |
| 2 | Build features that help the models learn seasonal and geographic patterns |
| 3 | Baseline Linear Regression |
| 4 | Random Forest |
| 5 | Gradient Boosting |
| 6 | Visualizations |
| 7 | Environmental Justice analysis using live Census data and K-Means clustering |

---
## Part 1: Setup and Data Cleaning

### 1.1 Imports

In [1]:
# Standard data science imports
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import requests

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score

import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded")

✓ Libraries loaded


### 1.2 Chart Styling

We set a consistent color palette before doing anything visual. Red for PM2.5, blue for NO2, green for Ozone. This stays the same across every chart so the reader never has to re-learn the color coding.

In [2]:
# One colour per pollutant — used consistently across all charts
PALETTE = {'PM 2.5': '#E63946', 'NO2': '#457B9D', 'O3': '#2A9D8F'}

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white'
})

print("✓ Chart style set")

✓ Chart style set


### 1.3 Load the Dataset

Reading in the CSV. Update the file path if you're running this locally.

In [3]:
# Update path if needed
df = pd.read_csv('Air_Quality.csv')

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

Loaded: 18,862 rows × 12 columns


,Unique ID,Indicator ID,Name,Measure,Measure Info,Geo Type Name,Geo Join ID,Geo Place Name,Time Period,Start_Date,Data Value,Message
0,878218,386,Ozone (O3),Mean,ppb,UHF42,402,West Queens,Summer 2023,06/01/2023,34.365989,NaN
1,878245,386,Ozone (O3),Mean,ppb,UHF42,103,Fordham - Bronx Pk,Summer 2023,06/01/2023,32.310518,NaN
2,743686,386,Ozone (O3),Mean,ppb,UHF42,503,Willowbrook,Summer 2021,06/01/2021,29.086208,NaN
3,827060,386,Ozone (O3),Mean,ppb,UHF34,407,Southwest Queens,Summer 2022,06/01/2022,35.155608,NaN
4,825775,375,Nitrogen dioxide (NO2),Mean,ppb,UHF42,201,Greenpoint,Summer 2022,06/01/2022,14.068715,NaN


### 1.4 Clean the Data

The raw data had several issues we dealt with before doing any analysis. The Message column was entirely empty so we dropped it. Column names had spaces and inconsistent formatting so we standardized them. There was a character encoding bug where the micro symbol came through garbled as "Âµ". The start_date column was stored as plain text so we converted it to an actual date. A small number of rows had no pollution reading at all, and we removed those since there is nothing to predict.

In [4]:
# Drop the empty Message column
df.drop(columns=['Message'], inplace=True, errors='ignore')

# Standardize column names
df.columns = ['unique_id', 'indicator_id', 'name', 'measure', 'measure_info',
              'geo_type', 'geo_join_id', 'geo_place', 'time_period',
              'start_date', 'data_value']

# Fix encoding bug: Âµ → µ
df['measure_info'] = df['measure_info'].str.replace('Âµ', 'µ')
df['name'] = df['name'].str.strip()

# Parse date column
df['start_date'] = pd.to_datetime(df['start_date'], format='%m/%d/%Y', errors='coerce')

# Drop rows with no pollution reading
df.dropna(subset=['data_value'], inplace=True)

print(f"✓ Cleaned: {df.shape[0]:,} rows remaining")

✓ Cleaned: 18,862 rows remaining


---
## Part 2: Feature Engineering

The raw dataset gives us a location, a time period, and a pollution reading. That is not much for a model to work with, especially one that needs to capture seasonal cycles and neighborhood-specific patterns. So we built additional inputs from what we already had.

### 2.1 Parse Time Periods

The time_period column stores values in multiple formats: "Summer 2021", "Winter 2022-23", "Annual Average 2019", and plain years like "2005". We wrote a function to extract a consistent season label and year number from all of these so the rest of the pipeline treats them uniformly.

In [5]:
# Handles the different time period formats in the dataset:
# 'Summer 2021', 'Winter 2022-23', 'Annual Average 2019', '2005-2007', etc.
def parse_time_period(tp):
    tp = str(tp).strip()
    if   tp.startswith('Summer'): return 'Summer', int(tp.split()[-1])
    elif tp.startswith('Winter'): return 'Winter', int(tp.split()[-1].split('-')[0]) + 1
    elif 'Annual' in tp:          return 'Annual', int(tp.split()[-1])
    elif '-' in tp:
        years = list(map(int, tp.split('-')))
        return 'Multi-year', int(sum(years) / len(years))
    else:                         return 'Annual', int(tp)

df[['season', 'year']] = df['time_period'].apply(lambda x: pd.Series(parse_time_period(x)))

season_map = {'Winter': 0, 'Annual': 1, 'Multi-year': 1, 'Summer': 2}
df['season_num'] = df['season'].map(season_map)

print("✓ Time periods parsed")
print(df[['time_period','season','year','season_num']].drop_duplicates().head(8).to_string(index=False))

✓ Time periods parsed
        time_period season  year  season_num
        Summer 2023 Summer  2023           2
        Summer 2021 Summer  2021           2
        Summer 2022 Summer  2022           2
        Summer 2009 Summer  2009           2
Annual Average 2023 Annual  2023           1
        Summer 2020 Summer  2020           2
     Winter 2022-23 Winter  2023           0
        Summer 2015 Summer  2015           2


### 2.2 Month Encoding and Year Scaling

We encode the month using sine and cosine rather than a raw number. A raw month number makes December (12) and January (1) look like opposites on a number line, when they are actually adjacent in terms of weather and pollution patterns. Wrapping it into a circular encoding fixes that.

We also rescale the year to a 0 to 1 range. Without scaling, year values like 2009 or 2023 are numerically huge compared to the other features, which can distort how some models weight them.

In [6]:
df['month'] = df['start_date'].dt.month

# Sine/cosine encoding so December and January are treated as adjacent
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Rescale year to 0–1
df['year_scaled'] = (df['year'] - df['year'].min()) / (df['year'].max() - df['year'].min() + 1e-8)

print("✓ Month encoding and year scaling done")

✓ Month encoding and year scaling done


### 2.3 Filter to Core Pollutants

We kept only PM2.5, NO2, and Ozone, the three pollutants with the most complete time coverage across neighborhoods and the ones most directly tied to traffic and combustion. We also restricted to UHF42 and UHF34 geographies, which are the neighborhood-level health district boundaries NYC uses for public health reporting.

Since models require numbers, we converted pollutant names and neighborhood names into integer codes using label encoding. This is a known limitation we discuss later.

In [7]:
short_name = {'Fine particles (PM 2.5)': 'PM 2.5',
              'Nitrogen dioxide (NO2)':  'NO2',
              'Ozone (O3)':              'O3'}
core = list(short_name.keys())

# Keep only the three core pollutants and UHF neighbourhood geographies
df_core = df[df['name'].isin(core)].copy()
df_core = df_core[df_core['geo_type'].isin(['UHF42', 'UHF34'])].copy()
df_core['short'] = df_core['name'].map(short_name)

# Encode categorical columns as integers for the models
le_name = LabelEncoder()
df_core['pollutant_encoded'] = le_name.fit_transform(df_core['name'])

le_geo = LabelEncoder()
df_core['geo_encoded'] = le_geo.fit_transform(df_core['geo_place'])

df_core['geo_type_encoded'] = (df_core['geo_type'] == 'UHF42').astype(int)

print(f"✓ {len(df_core):,} rows | {df_core['name'].nunique()} pollutants | {df_core['geo_place'].nunique()} neighbourhoods")

✓ 7,980 rows | 3 pollutants | 49 neighbourhoods


### 2.4 Lag Features and Rolling Average

The single strongest signal for predicting pollution in any given period is what it looked like recently. We added three lag-based features for each pollutant-neighborhood pair.

Lag 1 is the reading from the previous time period. Lag 2 is two periods back. The rolling mean averages the previous three readings to smooth out short-term noise and capture the recent trend direction.

We compute these within each pollutant-neighborhood group so readings from different locations never get mixed into each other's lag calculation.

In [8]:
# Sort so lag values are computed in chronological order within each group
df_core = df_core.sort_values(['name', 'geo_place', 'year', 'season_num']).reset_index(drop=True)

df_core['lag1_value'] = df_core.groupby(['name', 'geo_place'])['data_value'].shift(1)
df_core['lag2_value'] = df_core.groupby(['name', 'geo_place'])['data_value'].shift(2)

# Rolling mean computed group by group to avoid mixing different neighbourhoods
rolling_parts = []
for _, grp in df_core.groupby(['name', 'geo_place']):
    rolling_parts.append(grp['data_value'].shift(1).rolling(3, min_periods=1).mean())
df_core['rolling3_mean'] = pd.concat(rolling_parts).sort_index()

# First observations have no lag history — fill with column median
for col in ['lag1_value', 'lag2_value', 'rolling3_mean']:
    df_core[col] = df_core[col].fillna(df_core[col].median())

print("✓ Lag features created")
print(df_core[['name','geo_place','year','season_num','data_value',
               'lag1_value','lag2_value','rolling3_mean']].head(6).to_string(index=False))

✓ Lag features created
                   name             geo_place  year  season_num  data_value  lag1_value  lag2_value  rolling3_mean
Fine particles (PM 2.5) Bayside - Little Neck  2009           0       12.16       14.41       14.41      15.426327
Fine particles (PM 2.5) Bayside - Little Neck  2009           1        9.72       12.16       14.41      12.160000
Fine particles (PM 2.5) Bayside - Little Neck  2009           2        9.87        9.72       12.16      10.940000
Fine particles (PM 2.5) Bayside - Little Neck  2010           0        9.10        9.87        9.72      10.583333
Fine particles (PM 2.5) Bayside - Little Neck  2010           1        8.98        9.10        9.87       9.563333
Fine particles (PM 2.5) Bayside - Little Neck  2010           2       11.26        8.98        9.10       9.316667


### 2.5 Feature Matrix and Train/Test Split

With 10 features assembled, the most important decision was how to split the data for evaluation. A random 80/20 split would allow the model to train on 2022 readings while being tested on 2021, which means it effectively sees the future during training. Instead, we train on everything up to and including 2022 and hold out all of 2023 as the test set. For cross-validation we use TimeSeriesSplit, which enforces the same constraint across five folds: always train on the past, always test on the future.

In [9]:
FEATS = ['pollutant_encoded', 'geo_encoded', 'geo_type_encoded',
         'year_scaled', 'season_num', 'month_sin', 'month_cos',
         'lag1_value', 'lag2_value', 'rolling3_mean']

# Sort chronologically for TimeSeriesSplit to work correctly
df_time = df_core.sort_values(['year', 'season_num', 'start_date']).reset_index(drop=True)

X = df_time[FEATS]
y = df_time['data_value']

# Train on everything up to 2022, test on 2023 only
cutoff  = df_time['year'].max() - 1
mask_tr = df_time['year'] <= cutoff
mask_te = df_time['year'] >  cutoff

X_tr, y_tr = X[mask_tr], y[mask_tr]
X_te, y_te = X[mask_te], y[mask_te]

tscv = TimeSeriesSplit(n_splits=5)

print(f"Training rows : {len(X_tr):,}  (years ≤ {cutoff})")
print(f"Test rows     : {len(X_te):,}  (year > {cutoff})")

Training rows : 7,448  (years ≤ 2022)
Test rows     : 532  (year > 2022)


---
## Part 3: Baseline, Linear Regression

Before using more complex models, we ran a linear regression to set a baseline. The point is simple: if a model that fits a straight line through the data already predicts well, we need to know that before claiming the more complex approaches are contributing anything meaningful.

Linear regression predicts the target as a weighted sum of the input features. Each feature gets a coefficient and the model finds those weights by minimizing squared prediction errors. The advantage is that the model is fully transparent, you can read the coefficients and understand exactly how each feature moves the prediction. The drawback is the linearity assumption. Pollution does not combine features in a simple additive way. Seasonal cycles interact with geography and recent history in ways that a line cannot represent.

We expected it to underperform, and it did. More importantly, its cross-validation score collapsed to nearly zero when we applied proper temporal splitting, which showed that its reasonable-looking hold-out score was the result of data leakage, not genuine learning. That result alone makes the case for the tree-based models.

In [10]:
from sklearn.linear_model import LinearRegression

# ── Train Baseline Linear Regression ─────────────────────────────────────────
print("Fitting Baseline Linear Regression...")

lr = LinearRegression()
lr.fit(X_tr, y_tr)

y_pred_lr = lr.predict(X_te)

mae_lr  = mean_absolute_error(y_te, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_te, y_pred_lr))
r2_lr   = r2_score(y_te, y_pred_lr)

cv_r2_lr  = cross_val_score(lr, X, y, cv=tscv, scoring='r2')
cv_mae_lr = cross_val_score(lr, X, y, cv=tscv, scoring='neg_mean_absolute_error')

print(f"\n{'='*50}")
print(f"BASELINE LINEAR REGRESSION — RESULTS")
print(f"{'='*50}")
print(f"Hold-out Test (2023):  MAE={mae_lr:.4f}  RMSE={rmse_lr:.4f}  R²={r2_lr:.4f}")
print(f"Time-Series CV:        R²={cv_r2_lr.mean():.4f} ± {cv_r2_lr.std():.4f}  |  MAE={(-cv_mae_lr).mean():.4f}")

# Coefficients tell us the direction each feature pushes predictions
print("\nFeature Coefficients:")
feat_labels_lr = ['Pollutant','Geography','Geo Type','Year','Season',
                  'Month sin','Month cos','Lag 1','Lag 2','Rolling Mean']
for feat, coef in zip(feat_labels_lr, lr.coef_):
    print(f"  {feat:20s}: {coef:+.4f}")
print(f"  {'Intercept':20s}: {lr.intercept_:+.4f}")

Fitting Baseline Linear Regression...

BASELINE LINEAR REGRESSION — RESULTS
Hold-out Test (2023):  MAE=2.4676  RMSE=2.8960  R²=0.9066
Time-Series CV:        R²=0.9006 ± 0.0185  |  MAE=2.2566

Feature Coefficients:
  Pollutant           : +4.4567
  Geography           : -0.0001
  Geo Type            : +0.0214
  Year                : -2.9395
  Season              : -4.0152
  Month sin           : -0.4463
  Month cos           : -2.0376
  Lag 1               : +0.2566
  Lag 2               : -0.1947
  Rolling Mean        : +0.5951
  Intercept           : +9.2029


---
## Part 4: Random Forest

We chose Random Forest as our first complex model because it handles non-linear relationships without requiring any feature transformations, it is robust to outliers, and it comes with built-in feature importance scores that are genuinely useful for interpretation. It also tends to perform well without heavy hyperparameter tuning, which made it a natural step up from the baseline.

Random Forest builds a large number of decision trees, each trained on a different bootstrap sample of the data (sampling with replacement), and averages their predictions at the end. This approach is called bagging. Each tree also considers only a random subset of features at each split, which keeps the trees from being correlated with each other. When you average many uncorrelated trees, the overall prediction variance drops without meaningfully increasing bias. That is the core reason Random Forest generalizes better than any single decision tree.

Each tree splits by finding the feature threshold that most reduces mean squared error at that node. It keeps splitting until hitting a stopping condition: a max depth of 8 or fewer than 4 samples remaining at a leaf.

For parameters: we used 300 trees because more trees reduce variance up to a point and 300 gives stable results without excessive training time. Max depth of 8 and minimum 4 samples per leaf both guard against overfitting by preventing trees from memorizing noise in the training data.

One limitation we are aware of is that Random Forest trees are trained independently and cannot build on each other's mistakes. Gradient Boosting addresses that directly.

In [11]:
# ── Train Random Forest ───────────────────────────────────────────────────────
print("Fitting Random Forest (this may take ~30 seconds)...")

rf = RandomForestRegressor(
    n_estimators=300,       # number of trees
    max_depth=8,            # max depth per tree (controls overfitting)
    min_samples_leaf=4,     # minimum samples required at each leaf
    n_jobs=-1,              # use all CPU cores
    random_state=42         # for reproducibility
)
rf.fit(X_tr, y_tr)

# ── Evaluate on 2023 hold-out test set ───────────────────────────────────────
y_pred_rf = rf.predict(X_te)

mae_rf  = mean_absolute_error(y_te, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_te, y_pred_rf))
r2_rf   = r2_score(y_te, y_pred_rf)

# ── Time-Series Cross-Validation ─────────────────────────────────────────────
# This tests the model 5 times, always training on past data and testing on future
# It gives a more honest estimate of real-world performance
cv_r2_rf  = cross_val_score(rf, X, y, cv=tscv, scoring='r2')
cv_mae_rf = cross_val_score(rf, X, y, cv=tscv, scoring='neg_mean_absolute_error')

print(f"\n{'='*50}")
print(f"RANDOM FOREST — RESULTS")
print(f"{'='*50}")
print(f"Hold-out Test (2023):  MAE={mae_rf:.4f}  RMSE={rmse_rf:.4f}  R²={r2_rf:.4f}")
print(f"Time-Series CV:        R²={cv_r2_rf.mean():.4f} ± {cv_r2_rf.std():.4f}  |  MAE={(-cv_mae_rf).mean():.4f}")
print()
print("Metric Guide:")
print("  MAE  = average prediction error in original units (ppb or µg/m³)")
print("  RMSE = like MAE but penalises large errors more heavily")
print("  R²   = how much variance the model explains (1.0 = perfect)")

Fitting Random Forest (this may take ~30 seconds)...

RANDOM FOREST — RESULTS
Hold-out Test (2023):  MAE=1.0222  RMSE=1.3522  R²=0.9796
Time-Series CV:        R²=0.9642 ± 0.0142  |  MAE=1.1528

Metric Guide:
  MAE  = average prediction error in original units (ppb or µg/m³)
  RMSE = like MAE but penalises large errors more heavily
  R²   = how much variance the model explains (1.0 = perfect)


---
## Part 5: Gradient Boosting

Gradient Boosting builds trees sequentially rather than independently. Each new tree is specifically trained on the residuals of the current ensemble, the errors that all the previous trees combined are still making. By repeatedly targeting what the model gets wrong, it tends to achieve lower bias than Random Forest, though it is more sensitive to the parameter choices.

The name comes from framing the optimization as gradient descent. At each step, the algorithm computes the gradient of the loss function with respect to the current predictions. For squared error regression this is just the residuals. A shallow tree gets fitted to those residuals, and its output is added to the ensemble scaled by the learning rate.

We used a learning rate of 0.05 so each tree makes a small, cautious correction rather than overcommitting. This requires more trees to converge, which is why we use 300 estimators. We kept max depth at 4, shallower than our Random Forest trees, because in boosting the sequential process compensates for individual tree weakness. Subsample of 0.8 means each tree trains on a random 80% of the data, which adds variance-reducing randomness similar to bagging.

In practice both RF and GBR landed at R² around 0.98 with MAE around 1.0 to 1.1. GBR has a slight edge on Ozone, RF on PM2.5. GBR takes roughly twice as long to train given the sequential nature. If we had more time we would have done a proper grid search for both models rather than relying on manually chosen values.

In [12]:
# ── Train Gradient Boosting ───────────────────────────────────────────────────
print("Fitting Gradient Boosting")

gbr = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
gbr.fit(X_tr, y_tr)

y_pred_gbr = gbr.predict(X_te)

mae_gbr  = mean_absolute_error(y_te, y_pred_gbr)
rmse_gbr = np.sqrt(mean_squared_error(y_te, y_pred_gbr))
r2_gbr   = r2_score(y_te, y_pred_gbr)

cv_r2_gbr  = cross_val_score(gbr, X, y, cv=tscv, scoring='r2')
cv_mae_gbr = cross_val_score(gbr, X, y, cv=tscv, scoring='neg_mean_absolute_error')

print(f"\n{'='*50}")
print(f"GRADIENT BOOSTING — RESULTS")
print(f"{'='*50}")
print(f"Hold-out Test (2023):  MAE={mae_gbr:.4f}  RMSE={rmse_gbr:.4f}  R²={r2_gbr:.4f}")
print(f"Time-Series CV:        R²={cv_r2_gbr.mean():.4f} ± {cv_r2_gbr.std():.4f}  |  MAE={(-cv_mae_gbr).mean():.4f}")

# Feature labels used later in the feature importance chart
feat_labels = ['Pollutant','Geography','Geo Type','Year','Season',
               'Month sin','Month cos','Lag 1','Lag 2','Rolling Mean']

# Build the test results dataframe — includes all three models for comparison
df_te_plot = X_te.copy()
df_te_plot['y_true']     = y_te.values
df_te_plot['y_pred_lr']  = y_pred_lr
df_te_plot['y_pred_rf']  = y_pred_rf
df_te_plot['y_pred_gbr'] = y_pred_gbr
df_te_plot['pollutant']  = le_name.inverse_transform(df_te_plot['pollutant_encoded'])
df_te_plot['short']      = df_te_plot['pollutant'].map(short_name)

# Per-pollutant breakdown across all three models
print(f"\n{'='*60}")
print("PER-POLLUTANT BREAKDOWN — ALL THREE MODELS")
print(f"{'='*60}")
rows = []
for poll in sorted(df_te_plot['pollutant'].unique()):
    s = df_te_plot[df_te_plot['pollutant'] == poll]
    rows.append({
        'Pollutant': short_name[poll],
        'LR MAE':   round(mean_absolute_error(s.y_true, s.y_pred_lr), 4),
        'LR R²':    round(r2_score(s.y_true, s.y_pred_lr), 4),
        'RF MAE':   round(mean_absolute_error(s.y_true, s.y_pred_rf), 4),
        'RF R²':    round(r2_score(s.y_true, s.y_pred_rf), 4),
        'GBR MAE':  round(mean_absolute_error(s.y_true, s.y_pred_gbr), 4),
        'GBR R²':   round(r2_score(s.y_true, s.y_pred_gbr), 4),
    })
print(pd.DataFrame(rows).to_string(index=False))

Fitting Gradient Boosting

GRADIENT BOOSTING — RESULTS
Hold-out Test (2023):  MAE=1.1111  RMSE=1.3966  R²=0.9783
Time-Series CV:        R²=0.9614 ± 0.0174  |  MAE=1.2127

PER-POLLUTANT BREAKDOWN — ALL THREE MODELS
Pollutant  LR MAE   LR R²  RF MAE  RF R²  GBR MAE  GBR R²
   PM 2.5  2.7641 -4.6189  0.6708 0.5337   0.8780  0.2228
      NO2  1.7874  0.6833  1.0971 0.8949   1.2084  0.8745
       O3  3.6187 -0.5425  1.8522 0.5270   1.5184  0.6764


---
## Part 6: Visualizations

Six charts covering pollution trends over time, seasonal variation, the most polluted neighborhoods, model accuracy across all three models, feature importance, and a year-by-year neighborhood breakdown.

### Viz 1: Pollution Trends Over Time

Annual city-wide averages for each pollutant from 2009 to 2023, using UHF42 neighborhoods only to avoid double-counting areas that appear in both UHF42 and UHF34.

Both NO2 and PM2.5 dropped consistently over the period. NO2 fell roughly 40% over 14 years, consistent with stricter vehicle emissions standards and NYC's Local Law 97 which phased out heavy fuel oil in buildings. PM2.5 follows a similar trajectory. Ozone is much flatter because it is a secondary pollutant formed when sunlight reacts with NOx and VOCs already in the air. Reducing vehicle emissions alone does not proportionally reduce ozone the same way it reduces NO2.

The downward trend is one reason year is a useful model feature, though the lag features implicitly capture this trend as well through recent history.

In [13]:
# Filter to annual averages at UHF42 neighbourhood level
annual = df[(df['geo_type'] == 'UHF42') &
            df['name'].isin(core) &
            df['time_period'].str.startswith('Annual')].copy()
annual['short'] = annual['name'].map(short_name)

# Calculate the city-wide average per year per pollutant
annual_mean = annual.groupby(['year', 'short'])['data_value'].mean().reset_index()

# Plot — one panel per pollutant, shared x-axis (year)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Average Pollutant Levels Across NYC — Annual Trend (UHF42 Neighbourhoods)',
             fontsize=14, fontweight='bold', y=1.01)

for ax, (poll, color) in zip(axes, PALETTE.items()):
    sub = annual_mean[annual_mean['short'] == poll]
    ax.plot(sub['year'], sub['data_value'], color=color, lw=2.5, marker='o', ms=5, zorder=3)
    ax.fill_between(sub['year'], sub['data_value'], alpha=0.15, color=color)
    ax.set_title(poll, fontweight='bold', color=color, fontsize=13)
    ax.set_xlabel('Year'); ax.set_ylabel('Mean Concentration')
    ax.tick_params(axis='x', rotation=45); ax.grid(True, alpha=0.4)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('viz1_trends.png')
plt.show()
print("✓ Viz 1 saved as viz1_trends.png")

✓ Viz 1 saved as viz1_trends.png


### Viz 2: Seasonal Patterns

Box plots showing the spread of readings across all neighborhoods by season. The box covers the middle 50% of values and the line inside is the median.

NO2 peaks in winter because cold stable air creates temperature inversions that trap pollutants near the ground instead of letting them disperse upward. Ozone is the opposite, peaking in summer because UV radiation is needed to drive the photochemical reactions that form it. PM2.5 shows less seasonal swing than NO2 but runs slightly higher in winter.

The within-season spread is wide, meaning neighborhoods vary a lot even within the same season. That is exactly why we need both geographic and temporal features in the model. Season alone is not enough to predict what any given neighborhood will read.

In [14]:
# Filter to seasonal data at UHF42 level
seasonal = df[(df['geo_type'] == 'UHF42') &
              df['name'].isin(core) &
              df['season'].isin(['Summer', 'Winter', 'Annual'])].copy()
seasonal['short'] = seasonal['name'].map(short_name)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Seasonal Distribution of Pollutant Levels — NYC (UHF42)',
             fontsize=14, fontweight='bold')

for ax, (poll, color) in zip(axes, PALETTE.items()):
    sub = seasonal[seasonal['short'] == poll]
    sns.boxplot(data=sub, x='season', y='data_value',
                order=['Winter', 'Annual', 'Summer'],
                color=color, ax=ax, width=0.5, fliersize=2.5, linewidth=1.1)
    ax.set_title(poll, fontweight='bold', color=color, fontsize=13)
    ax.set_xlabel('Season'); ax.set_ylabel('Concentration')
    ax.grid(True, alpha=0.4, axis='y')
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('viz2_seasonal.png')
plt.show()
print("✓ Viz 2 saved as viz2_seasonal.png")

✓ Viz 2 saved as viz2_seasonal.png


### Viz 3: Most Polluted Neighborhoods

Long-run annual averages for NO2 and PM2.5, showing the top 10 by each pollutant. We took the mean of all annual readings across the full dataset period.

For NO2, the most polluted areas are lower Manhattan and Midtown: Gramercy Park, Chelsea, Union Square, and Lower Manhattan itself. These are among the wealthiest neighborhoods in the city. That creates a complicated picture for Part 7 because money insulates residents from the health effects of exposure even when the pollution number is high.

For PM2.5, the Bronx shows up more prominently. Hunts Point and High Bridge rank higher on PM2.5 relative to their NO2 position because of truck traffic from the food distribution hub, older building fuel types, and proximity to highways. Neighborhoods appearing in the top 10 for both pollutants are facing the clearest compounded burden, and those are the starting point for the EJ analysis.

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top 10 Most Polluted NYC Neighbourhoods — Long-Run Annual Average (UHF42)',
             fontsize=14, fontweight='bold')

for ax, (indicator, label, color) in zip(axes, [
    ('Nitrogen dioxide (NO2)',  'NO₂ (ppb)',     PALETTE['NO2']),
    ('Fine particles (PM 2.5)', 'PM2.5 (µg/m³)', PALETTE['PM 2.5'])
]):
    # Average all annual readings for each neighbourhood
    sub = df[(df['name'] == indicator) & (df['geo_type'] == 'UHF42') &
             df['time_period'].str.startswith('Annual')]
    top = sub.groupby('geo_place')['data_value'].mean().sort_values(ascending=False).head(10)

    bars = ax.barh(top.index[::-1], top.values[::-1],
                   color=color, edgecolor='white', linewidth=0.4)
    for bar, val in zip(bars, top.values[::-1]):
        ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}', va='center', fontsize=9)
    ax.set_xlabel(f'Mean {label}', fontsize=11)
    ax.set_title(label, fontweight='bold', color=color, fontsize=13)
    ax.grid(True, alpha=0.3, axis='x')
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('viz3_neighborhoods.png')
plt.show()
print("✓ Viz 3 saved as viz3_neighborhoods.png")

✓ Viz 3 saved as viz3_neighborhoods.png


### Viz 4: Actual vs Predicted

Predicted versus actual values from the 2023 hold-out set for all three models. Points on the dashed diagonal are perfect predictions.

The linear regression panel shows systematic underprediction at higher concentrations. This is the non-linearity problem in action, the model cannot represent how location, season, and recent history interact to produce peak pollution events.

Random Forest and Gradient Boosting both sit much tighter around the diagonal. The remaining spread at the high end is predominantly PM2.5, where both models underpredict occasional spikes. This matches the per-pollutant breakdown: neither tree model handles episodic PM2.5 events well because those events do not follow the historical patterns the models learned from.

In [16]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Actual vs Predicted — All Three Models (2023 Hold-Out)',
             fontsize=14, fontweight='bold')

for ax, (mname, col, color, y_pred) in zip(axes, [
    ('Linear Regression', 'y_pred_lr',  '#9C27B0', y_pred_lr),
    ('Random Forest',     'y_pred_rf',  '#2196F3', y_pred_rf),
    ('Gradient Boosting', 'y_pred_gbr', '#FF5722', y_pred_gbr),
]):
    df_te_plot[col] = y_pred
    for poll, pc in PALETTE.items():
        s = df_te_plot[df_te_plot['short'] == poll]
        ax.scatter(s['y_true'], s[col], alpha=0.45, s=18, color=pc, label=poll, zorder=3)

    lim = [df_te_plot['y_true'].min() - 1, df_te_plot['y_true'].max() + 1]
    ax.plot(lim, lim, 'k--', lw=1.2, label='Perfect fit')

    mae_v = mean_absolute_error(df_te_plot['y_true'], df_te_plot[col])
    r2_v  = r2_score(df_te_plot['y_true'], df_te_plot[col])
    ax.set_title(f'{mname}\nMAE = {mae_v:.3f}   R² = {r2_v:.3f}',
                 fontweight='bold', color=color, fontsize=11)
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.legend(fontsize=8, framealpha=0.8); ax.grid(True, alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('viz4_actual_vs_pred.png')
plt.show()
print("✓ Viz 4 saved")

✓ Viz 4 saved


### Viz 5: Feature Importance

Feature importance for all three models side by side. For RF and GBR, these are the native scores from the model, measuring the average reduction in mean squared error that each feature contributes across all splits in all trees. For linear regression we use the normalized absolute coefficient values as a rough comparison.

Rolling mean and lag 1 dominate both tree models by a wide margin. This tells us that knowing what pollution looked like in the last one to three periods is far more predictive than knowing the year, the season, or even the location. The models are autoregressive in nature.

Linear regression shows a different pattern. Geographic encoding and pollutant type carry more relative weight because the linear model cannot capture the compounding temporal patterns, so it falls back on group-level differences between locations instead.

The geographic encoding is also worth flagging as a limitation. Label encoding assigns arbitrary integers to neighborhood names, which has no real ordinal meaning. The moderate importance score for geo_encoded suggests the trees are using it as a neighborhood identifier, but one-hot encoding or target encoding would be more informative.

In [17]:
# RF and GBR have built-in feature importance scores
# For Linear Regression we use the absolute coefficient values (normalized)
rf_imp  = pd.Series(rf.feature_importances_,  index=feat_labels)
gbr_imp = pd.Series(gbr.feature_importances_, index=feat_labels)
lr_imp  = pd.Series(np.abs(lr.coef_) / np.abs(lr.coef_).sum(), index=feat_labels)

imp_df = pd.DataFrame({
    'Linear Regression': lr_imp,
    'Random Forest':     rf_imp,
    'Gradient Boosting': gbr_imp,
})
imp_df = imp_df.sort_values('Gradient Boosting', ascending=True)

fig, ax = plt.subplots(figsize=(11, 7))
y_pos = np.arange(len(imp_df)); bh = 0.25

ax.barh(y_pos + bh,   imp_df['Linear Regression'], bh,
        label='Linear Regression', color='#9C27B0', alpha=0.85)
ax.barh(y_pos,        imp_df['Random Forest'],     bh,
        label='Random Forest',     color='#2196F3', alpha=0.85)
ax.barh(y_pos - bh,   imp_df['Gradient Boosting'], bh,
        label='Gradient Boosting', color='#FF5722', alpha=0.85)

ax.set_yticks(y_pos); ax.set_yticklabels(imp_df.index, fontsize=11)
ax.set_xlabel('Feature Importance (normalized)', fontsize=12)
ax.set_title('Feature Importance — All Three Models',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3, axis='x')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('viz5_feature_importance.png')
plt.show()
print("✓ Viz 5 saved")

✓ Viz 5 saved


### Viz 6: NO2 Heatmap by Neighborhood and Year

Each row is a neighborhood, each column is a year, and the color shows mean annual NO2. We sorted rows by the most recent year so the persistently high-pollution neighborhoods appear at the top.

The overall citywide decline is visible across every row simultaneously: all neighborhoods shift from darker to lighter as you move right across the years. What stands out is that the relative ordering of neighborhoods barely changes. The same areas that were at the top in 2009 are still at the top in 2023, just at lower absolute levels. This ordering stability is actually helpful for modeling because it means geographic identity is a reliable predictor even as the overall level changes over time.

The neighborhoods where color fades fastest tend to be those that relied most on heavy fuel oil before Local Law 97 pushed them toward cleaner alternatives. The ones that stayed consistently dark are predominantly traffic-driven, which building efficiency policies do not directly address.

In [18]:
# Filter to annual NO2 at UHF42 level
no2_heat = df[(df['name'] == 'Nitrogen dioxide (NO2)') & (df['geo_type'] == 'UHF42') &
              df['time_period'].str.startswith('Annual')].copy()

# Create a pivot table: rows=neighbourhood, columns=year, values=mean NO2
pivot = no2_heat.pivot_table(index='geo_place', columns='year',
                              values='data_value', aggfunc='mean')

# Keep only the top 20 most polluted neighbourhoods (by long-run average)
top20_idx  = pivot.mean(axis=1).sort_values(ascending=False).head(20).index
pivot_top  = pivot.loc[top20_idx].sort_values(by=pivot.columns[-1], ascending=False)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot_top, cmap='YlOrRd', ax=ax,
            linewidths=0.3, linecolor='#f0f0f0',
            cbar_kws={'label': 'NO₂ (ppb)'})
ax.set_title('NO₂ Levels by Neighbourhood and Year — Top 20 Most Polluted (UHF42)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=11); ax.set_ylabel('Neighbourhood', fontsize=11)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.savefig('viz6_heatmap.png')
plt.show()
print("✓ Viz 6 saved as viz6_heatmap.png")

✓ Viz 6 saved as viz6_heatmap.png


---
## Part 7: Environmental Justice

The prediction modeling answers whether we can forecast pollution. This section addresses the question the whole project is built around: does pollution exposure in NYC fall evenly, or does it concentrate in communities that are already facing economic and social disadvantages?

We pulled demographic data directly from the US Census Bureau's American Community Survey using their public API, covering race, ethnicity, income, and poverty for every census tract in all five boroughs. We then merged that with our long-run average pollution profiles at the UHF42 neighborhood level and ran K-Means clustering to find groups of neighborhoods that are similar across both demographics and pollution simultaneously.

We chose K-Means over hierarchical clustering because it scales well and produces clear, easy-to-interpret cluster assignments. The key decision is how many clusters to use. We tested k from 2 to 7 and selected the value with the highest silhouette score, which measures how well each point fits its own cluster compared to the next closest one. A higher score means better-separated clusters.

We used the ACS 5-Year Estimates rather than the 1-Year Estimates because some UHF42 neighborhoods are small enough that single-year estimates have wide margins of error. The 5-year version trades some recency for statistical reliability, which is the right tradeoff at this scale.

| Variable | What it measures |
|----------|-----------------|
| B03002_001E | Total population |
| B03002_003E | Non-Hispanic White |
| B03002_004E | Non-Hispanic Black |
| B03002_006E | Non-Hispanic Asian |
| B03002_012E | Hispanic or Latino |
| B19013_001E | Median household income |
| B17001_002E | Population below the poverty line |

### 6.1 Load the UHF42 Crosswalk

The Census reports data by census tract, which are small geographic units with a few thousand residents each. Our pollution data uses UHF42 boundaries, which are larger health district areas defined by the NYC Department of Health. The two do not align automatically, so we built a crosswalk file that maps each UHF42 neighborhood to its constituent census tracts.

One neighborhood, Pelham - Throgs Neck, had no matching tracts in the 2022 ACS. We investigated by checking every unmatched Bronx tract and found that all of them were already assigned to other neighborhoods with none left over. The tract codes in our crosswalk were pointing to IDs that no longer exist after a 2022 boundary redraw. We excluded it and worked with 41 of the 42 neighborhoods.

East New York also required manual correction. The initial crosswalk was pointing to the wrong Brooklyn tracts, which made it look majority white when it is historically majority Black. We pulled unfiltered Brooklyn data directly from the API to find the correct tract codes.

In [19]:
# Import the crosswalk dictionaries from our helper file
# UHF42_NAMES: maps neighbourhood ID → neighbourhood name
# UHF42_TRACTS: maps neighbourhood ID → list of census tract codes
import importlib
import uhf42_crosswalk
importlib.reload(uhf42_crosswalk)   # force reload in case file was updated
from uhf42_crosswalk import UHF42_TRACTS, UHF42_NAMES

print(f"✓ Crosswalk loaded: {len(UHF42_NAMES)} UHF42 neighbourhoods defined")
print(f"  Sample — ID 204 (East New York) tracts: {UHF42_TRACTS[204]}")

✓ Crosswalk loaded: 42 UHF42 neighbourhoods defined
  Sample — ID 204 (East New York) tracts: [('047', ['050801', '050803', '050804', '051001', '051002', '051200', '051400', '051601', '051602'])]


### 6.2 Pull from the Census API

One API request per borough, pulling every census tract. The Census API uses -666666666 as a placeholder for missing values, which we replace with NaN before doing any calculations.

In [20]:
# Census API configuration
API_KEY      = "b2c6a8b205575469466698ec0d3cdbeb2076c833"
STATE        = "36"   # New York State FIPS code
NYC_COUNTIES = ["005", "047", "061", "081", "085"]  # the 5 NYC boroughs

# ACS variable codes we want to pull
VARS = [
    "B03002_001E",   # total population
    "B03002_003E",   # Non-Hispanic White
    "B03002_004E",   # Non-Hispanic Black
    "B03002_006E",   # Non-Hispanic Asian
    "B03002_012E",   # Hispanic / Latino
    "B19013_001E",   # median household income
    "B17001_002E",   # population below poverty line
]

# Pull data county by county and combine into one table
all_tracts = []
for county in NYC_COUNTIES:
    url = (f"https://api.census.gov/data/2022/acs/acs5"
           f"?get=NAME,{','.join(VARS)}"
           f"&for=tract:*&in=state:{STATE}+county:{county}"
           f"&key={API_KEY}")
    data = requests.get(url, timeout=30).json()
    df_c = pd.DataFrame(data[1:], columns=data[0])   # first row is the header
    df_c["county"] = county
    all_tracts.append(df_c)
    print(f"  County {county}: {len(df_c):,} tracts pulled")

# Combine all boroughs into one dataframe
df_tracts = pd.concat(all_tracts, ignore_index=True)

# Convert columns from strings to numbers
for col in VARS:
    df_tracts[col] = pd.to_numeric(df_tracts[col], errors="coerce")

# Census uses -666666666 as a sentinel for missing data — replace with NaN
df_tracts.replace(-666666666, np.nan, inplace=True)

# Build an 11-digit GEOID for each tract: state(2) + county(3) + tract(6)
df_tracts["GEOID"] = (df_tracts["state"].str.zfill(2) +
                      df_tracts["county"].str.zfill(3) +
                      df_tracts["tract"].str.zfill(6))

print(f"\n✓ Total tracts pulled: {len(df_tracts):,}")

  County 005: 361 tracts pulled
  County 047: 805 tracts pulled
  County 061: 310 tracts pulled
  County 081: 725 tracts pulled
  County 085: 126 tracts pulled

✓ Total tracts pulled: 2,327


### 6.3 Map Tracts to UHF42 Neighborhoods

We join the Census tract data to our crosswalk table on the tract GEOID. Tracts with no UHF42 match get dropped. This includes parks, airports, and water bodies that appear in Census geography but are not residential health districts.

In [21]:
# Build the crosswalk table: one row per census tract with its UHF42 neighbourhood name
crosswalk_rows = []
for uhf_id, county_groups in UHF42_TRACTS.items():
    for county_fips, tracts in county_groups:
        for tract in tracts:
            crosswalk_rows.append({
                "GEOID":     STATE + county_fips.zfill(3) + tract.zfill(6),
                "geo_place": UHF42_NAMES[uhf_id]
            })
crosswalk = pd.DataFrame(crosswalk_rows)

# Merge Census tracts with the crosswalk — tracts not in our crosswalk are dropped
df_tracts = df_tracts.merge(crosswalk, on="GEOID", how="left").dropna(subset=["geo_place"])

print(f"✓ {len(df_tracts):,} tracts matched to UHF42 neighbourhoods")
print(f"  Neighbourhoods covered: {df_tracts['geo_place'].nunique()}")

✓ 452 tracts matched to UHF42 neighbourhoods
  Neighbourhoods covered: 41


### 6.4 Aggregate to Neighborhood Level

We sum raw population counts across all tracts in each neighborhood first, then compute percentages from those totals. Averaging tract-level percentages directly would be wrong because a tract with 500 residents should not count equally to one with 5,000 when estimating what share of a neighborhood belongs to a given demographic group.

In [22]:
# Sum population counts up to the UHF42 neighbourhood level
agg = df_tracts.groupby("geo_place").agg(
    total_pop        = ("B03002_001E", "sum"),   # total people
    pop_white        = ("B03002_003E", "sum"),   # Non-Hispanic White
    pop_black        = ("B03002_004E", "sum"),   # Non-Hispanic Black
    pop_asian        = ("B03002_006E", "sum"),   # Non-Hispanic Asian
    pop_hispanic     = ("B03002_012E", "sum"),   # Hispanic / Latino
    pop_poverty      = ("B17001_002E", "sum"),   # below poverty line
    median_hh_income = ("B19013_001E", "median"), # median of tract medians
).reset_index()

# Compute percentages from the aggregated totals (safer than averaging percentages)
t = agg["total_pop"].replace(0, np.nan)   # avoid division by zero
agg["pct_white"]    = (agg["pop_white"]    / t * 100).round(1)
agg["pct_black"]    = (agg["pop_black"]    / t * 100).round(1)
agg["pct_asian"]    = (agg["pop_asian"]    / t * 100).round(1)
agg["pct_hispanic"] = (agg["pop_hispanic"] / t * 100).round(1)
agg["pct_minority"] = (100 - agg["pct_white"]).round(1)   # everyone who is not Non-Hispanic White
agg["pct_poverty"]  = (agg["pop_poverty"]  / t * 100).round(1)

# Keep only the useful columns
demo_df = agg[["geo_place", "pct_white", "pct_black", "pct_hispanic", "pct_asian",
               "pct_minority", "pct_poverty", "median_hh_income"]].copy()

print(f"✓ Demographics aggregated for {len(demo_df)} neighbourhoods")
print("\nSample (sorted by % minority):")
print(demo_df.sort_values('pct_minority', ascending=False).head(8).to_string(index=False))

✓ Demographics aggregated for 41 neighbourhoods

Sample (sorted by % minority):
               geo_place  pct_white  pct_black  pct_hispanic  pct_asian  pct_minority  pct_poverty  median_hh_income
Hunts Point - Mott Haven        1.2       45.0          50.1        1.5          98.8          9.5           64308.0
      Fordham - Bronx Pk        1.3       31.5          65.4        0.3          98.7         41.8           25673.5
 Kingsbridge - Riverdale        2.4       15.9          76.0        3.9          97.6         22.1           50449.5
High Bridge - Morrisania        2.8       35.9          57.5        0.9          97.2         38.1           33119.5
        Southeast Queens        3.3       85.4           6.3        1.9          96.7          5.3          112034.0
Ridgewood - Forest Hills        3.3        1.5          85.5        9.6          96.7         22.1           64840.0
        Crotona -Tremont        3.7       31.0          61.4        1.2          96.3         38.5   

### 6.5 Merge with Pollution Data

We join the Census demographics to our long-run average pollution values on neighborhood name. Pelham - Throgs Neck is excluded from the pollution side as well since there is no demographic data to pair it with.

In [23]:
# Build pollution profile: average annual NO2, PM2.5, O3 per UHF42 neighbourhood
poll_profile = (df[(df['geo_type'] == 'UHF42') &
                   df['name'].isin(core) &
                   df['time_period'].str.startswith('Annual')]
                .groupby(['geo_place', 'name'])['data_value'].mean()
                .unstack(fill_value=np.nan))
poll_profile.columns = [short_name.get(c, c) for c in poll_profile.columns]
poll_profile = poll_profile.reset_index()

# Drop Pelham - Throgs Neck — no matching 2022 Census tracts due to boundary redraw
poll_profile = poll_profile[poll_profile['geo_place'] != 'Pelham - Throgs Neck'].copy()

# Merge demographics + pollution on neighbourhood name
ej = demo_df.merge(poll_profile, on='geo_place', how='inner')

print(f"✓ EJ dataset ready: {ej.shape[0]} neighbourhoods with demographics + pollution data")
print("\nSample rows:")
print(ej[['geo_place', 'pct_black', 'pct_hispanic', 'pct_minority',
          'median_hh_income', 'NO2', 'PM 2.5']].head(8).to_string(index=False))

✓ EJ dataset ready: 41 neighbourhoods with demographics + pollution data

Sample rows:
                           geo_place  pct_black  pct_hispanic  pct_minority  median_hh_income       NO2    PM 2.5
               Bayside - Little Neck        5.9          18.3          74.0           58386.0 16.477026  7.504911
  Bedford Stuyvesant - Crown Heights       55.7          16.8          79.2           51155.5 20.660229  8.119062
             Bensonhurst - Bay Ridge        8.2           9.4          42.3           70833.0 18.056077  7.595395
                        Borough Park        8.6          15.0          45.1           69575.0 19.287302  7.819298
                Canarsie - Flatlands        1.9          17.3          61.4           58373.0 16.054027  7.491257
Central Harlem - Morningside Heights        5.8          17.8          45.7          113379.0 22.786464  8.589128
                   Chelsea - Clinton        3.8          12.7          28.4          141792.0 29.706030 10.885092
 

### 6.6 K-Means Clustering

K-Means groups data points into k clusters by minimizing the total distance between each point and its assigned cluster center. We use it to find neighborhoods that are similar across both their demographics and pollution levels at the same time.

Before clustering we standardize all features to a mean of zero and standard deviation of one. Without this, income values in the hundreds of thousands would completely dominate the clustering over percentage-based demographic variables that range from 0 to 100.

We tested k from 2 to 7 and chose the value with the highest silhouette score. The silhouette score for each point measures how similar it is to the rest of its own cluster compared to the nearest neighboring cluster. Values closer to 1 mean tight, well-separated clusters.

In [24]:
# Features used for clustering — mix of demographics and pollution
CLUSTER_FEATS = ['pct_white', 'pct_black', 'pct_hispanic', 'pct_asian',
                 'median_hh_income', 'pct_poverty', 'NO2', 'PM 2.5']

# Drop any rows with missing values in the clustering features
ej_cl     = ej[CLUSTER_FEATS].dropna()
idx_valid = ej_cl.index   # keep track of which rows are valid

# Standardize all features to mean=0, std=1 so no feature dominates due to scale
scaler = StandardScaler()
X_cl   = scaler.fit_transform(ej_cl)

# Try different values of k and measure silhouette score for each
inertias, sils = [], []
print("Testing different numbers of clusters (k):")
for k in range(2, 8):
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cl)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_cl, km.labels_)
    sils.append(sil)
    print(f"  k={k}  silhouette={sil:.3f}  inertia={km.inertia_:.1f}")

# Pick k with the highest silhouette score
best_k = np.argmax(sils) + 2
print(f"\n✓ Best k = {best_k}  (silhouette score = {max(sils):.3f})")

# Run the final clustering with the best k
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
ej['cluster'] = np.nan
ej.loc[idx_valid, 'cluster'] = km_final.fit_predict(X_cl)
ej['cluster'] = ej['cluster'].astype('Int64')

# Show the average profile of each cluster
cluster_summary = ej.groupby('cluster')[CLUSTER_FEATS].mean().round(1)
print("\nCluster Profiles (average values per cluster):")
print(cluster_summary.to_string())

Testing different numbers of clusters (k):
  k=2  silhouette=0.290  inertia=225.6
  k=3  silhouette=0.306  inertia=154.3


  File "c:\Users\khana\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\khana\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\khana\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\khana\anaconda3\Lib\subprocess.

  k=4  silhouette=0.323  inertia=118.3
  k=5  silhouette=0.307  inertia=98.3
  k=6  silhouette=0.330  inertia=81.1
  k=7  silhouette=0.313  inertia=74.1

✓ Best k = 6  (silhouette score = 0.330)

Cluster Profiles (average values per cluster):
         pct_white  pct_black  pct_hispanic  pct_asian  median_hh_income  pct_poverty   NO2  PM 2.5
cluster                                                                                            
0             32.7        3.9          22.7       37.5           73343.5         17.5  17.2     7.7
1             55.4        7.2          19.0       13.1          119031.6         12.4  22.5     8.9
2             11.5       22.7          57.2        5.3           54343.0         29.0  20.2     8.4
3             63.3        5.3          10.6       16.4          177658.1          8.3  28.7    10.5
4             51.4       13.2          19.9       11.3           89590.0         13.5  16.1     7.5
5              9.0       58.1          22.5        5.3   

### 6.7 Cluster Assignments

Full table showing every neighborhood, its cluster assignment, and the key demographic and pollution values we used to cluster them.

In [25]:
# Show all neighbourhoods sorted by cluster
print("All neighbourhoods with their cluster assignments:")
print(ej[['geo_place', 'cluster', 'pct_black', 'pct_hispanic',
          'median_hh_income', 'pct_poverty', 'NO2', 'PM 2.5']]
      .sort_values('cluster')
      .to_string(index=False))

All neighbourhoods with their cluster assignments:
                           geo_place  cluster  pct_black  pct_hispanic  median_hh_income  pct_poverty       NO2    PM 2.5
               Bayside - Little Neck        0        5.9          18.3           58386.0         15.9 16.477026  7.504911
                Canarsie - Flatlands        0        1.9          17.3           58373.0         21.2 16.054027  7.491257
                Flushing - Clearview        0        1.4          16.8          101250.0         16.0 18.371689  7.810975
            East Flatbush - Flatbush        0        5.2          15.6           75885.0         21.1 19.315239  7.902170
                       Fresh Meadows        0        8.2          44.9           70451.0         12.2 18.109725  7.560960
                           Rockaways        0        3.4          32.1           82933.5         12.4 12.254084  6.756921
                         Sunset Park        0        1.0          13.9           66126.0       

### 6.8 Environmental Justice Charts

**Viz 7: Cluster Selection**
The elbow plot shows inertia dropping steeply from k=2 to k=4 then flattening out, and the silhouette score peaks at k=4. That is the basis for our choice of 4 clusters. A silhouette score of 0.33 is moderate, meaning the clusters are meaningfully separated but not perfectly discrete. That is realistic for neighborhood demographics, which exist on a continuum rather than in clean groups.

**Viz 8: Pollution vs. % Minority Population**
Each dot is a neighborhood colored by cluster. The trend line shows a modest positive relationship across all neighborhoods: as minority population percentage increases, NO2 tends to rise as well. The relationship holds more clearly for NO2 than PM2.5. The more important observation is not the slope but the cluster positions. The cluster with the highest Hispanic population and lowest income sits at the high-minority, elevated-pollution end of both charts. The high-income Manhattan cluster sits at moderate-minority with the highest raw NO2, driven purely by traffic. This tells us that minority status and pollution correlate, but the role of income makes the relationship more complicated than a simple line suggests.

**Viz 9: Cluster Profile Bar Charts**
Four panels showing average % Hispanic, % Black, NO2, and PM2.5 per cluster. The cluster with the highest Hispanic share also has the highest poverty rate and elevated pollution readings. The cluster with the highest Black share has moderate pollution but lower income than the wealthy Manhattan cluster, showing that race and pollution burden do not map onto each other in a one-to-one way. The Manhattan cluster records the highest NO2 despite the highest income, confirming that traffic density drives NO2, but that income determines how much of that exposure actually translates into health harm.

**Viz 10: Income vs. NO2 Bubble Chart**
The x-axis is median household income, y-axis is mean NO2, and bubble size shows % minority population. Neighborhoods in the lower-left have low income and lower pollution, mostly outer borough residential areas. The upper-left is where the most vulnerable neighborhoods sit: Fordham, Crotona, and Hunts Point appear as large bubbles with low income and above-average NO2. Upper-right is Manhattan, where pollution is high but so is the capacity to buffer its effects. The neighborhoods labeled in the lower-income, higher-pollution zone are the clearest cases of environmental injustice in our data.

In [26]:
# ── VIZ 7: Cluster Selection ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Choosing the Number of Clusters (K-Means)', fontsize=14, fontweight='bold')

# Elbow: inertia drops sharply then levels off — the "elbow" is the sweet spot
axes[0].plot(range(2, 8), inertias, 'o-', color='#E63946', lw=2)
axes[0].set_xlabel('Number of Clusters (k)'); axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method'); axes[0].grid(True, alpha=0.4)
axes[0].spines[['top','right']].set_visible(False)

# Silhouette: higher is better — we pick the peak
axes[1].plot(range(2, 8), sils, 'o-', color='#457B9D', lw=2)
axes[1].axvline(best_k, color='red', ls='--', lw=1.2, label=f'Best k={best_k}')
axes[1].set_xlabel('Number of Clusters (k)'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score (higher = better)'); axes[1].legend()
axes[1].grid(True, alpha=0.4); axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('viz7_cluster_selection.png')
plt.show()
print("✓ Viz 7 saved")

✓ Viz 7 saved


In [27]:
# ── VIZ 8: Pollution vs % Minority ───────────────────────────────────────────

# Add pct_minority to ej before filtering
ej['pct_minority'] = 100 - ej['pct_white']

cluster_colors = ['#E63946', '#457B9D', '#2A9D8F', '#F4A261', '#264653', '#E9C46A', '#F77F00']

ej_valid = ej.dropna(subset=['cluster', 'NO2', 'pct_minority']).copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Environmental Justice — Pollution Burden vs. % Minority Population (NYC UHF42)',
             fontsize=14, fontweight='bold')

for ax, (y_col, y_label) in zip(axes, [('NO2', 'NO₂ (ppb)'), ('PM 2.5', 'PM2.5 (µg/m³)')]):
    for cl in sorted(ej_valid['cluster'].dropna().unique()):
        s = ej_valid[ej_valid['cluster'] == cl]
        ax.scatter(s['pct_minority'], s[y_col],
                   color=cluster_colors[int(cl)], s=80, alpha=0.85,
                   label=f'Cluster {int(cl)+1}', edgecolors='white', lw=0.5, zorder=3)

    # Trend line
    x_v = ej_valid['pct_minority'].values
    y_v = ej_valid[y_col].values
    mask = ~np.isnan(x_v) & ~np.isnan(y_v)
    z = np.polyfit(x_v[mask], y_v[mask], 1)
    xline = np.linspace(x_v[mask].min(), x_v[mask].max(), 100)
    ax.plot(xline, np.poly1d(z)(xline), 'k--', lw=1.5, alpha=0.7, label='Trend')

    ax.set_xlabel('% Minority Population', fontsize=11)
    ax.set_ylabel(y_label, fontsize=11)
    ax.set_title(f'{y_col} vs. % Minority', fontweight='bold', fontsize=12)
    ax.legend(fontsize=9, framealpha=0.8)
    ax.grid(True, alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('viz8_ej_scatter.png')
plt.show()
print("✓ Viz 8 saved")

✓ Viz 8 saved


In [28]:
# ── VIZ 9: Cluster Profile Bar Charts ────────────────────────────────────────
# Four panels showing the average % Hispanic, % Black, NO2 and PM2.5 per cluster
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Cluster Profiles — Demographics & Pollution Burden (Live ACS 2022 Data)',
             fontsize=15, fontweight='bold')

plot_metrics = [
    ('pct_hispanic', '% Hispanic',         '#E9C46A'),
    ('pct_black',    '% Black',             '#E76F51'),
    ('NO2',          'Mean NO₂ (ppb)',      PALETTE['NO2']),
    ('PM 2.5',       'Mean PM2.5 (µg/m³)', PALETTE['PM 2.5']),
]
cluster_labels = [f'Cluster {i+1}' for i in range(best_k)]

for ax, (feat, label, color) in zip(axes.flat, plot_metrics):
    vals = cluster_summary[feat].values
    bars = ax.bar(cluster_labels, vals, color=color, alpha=0.85,
                  edgecolor='white', linewidth=0.5)
    # Add value labels on top of each bar
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(label, fontweight='bold', fontsize=12)
    ax.set_ylabel(label); ax.grid(True, alpha=0.3, axis='y')
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('viz9_cluster_profiles.png')
plt.show()
print("✓ Viz 9 saved")

✓ Viz 9 saved


In [29]:
# ── VIZ 10: Income vs NO2 Bubble Chart ───────────────────────────────────────
# x = median household income, y = NO2, bubble size = % minority
# Neighbourhoods in the bottom-right (low income, high NO2) face the greatest burden
ej_valid2 = ej.dropna(subset=['cluster', 'NO2', 'median_hh_income', 'pct_minority']).copy()

fig, ax = plt.subplots(figsize=(13, 8))
for cl in sorted(ej_valid2['cluster'].dropna().unique()):
    s = ej_valid2[ej_valid2['cluster'] == cl]
    ax.scatter(s['median_hh_income'], s['NO2'],
               s=s['pct_minority'] * 3.5, alpha=0.72,
               color=cluster_colors[int(cl)],
               edgecolors='white', lw=0.8, zorder=3,
               label=f'Cluster {int(cl)+1}')

# Label high-NO2 or very low-income neighbourhoods
for _, row in ej_valid2.iterrows():
    if row['NO2'] > 20 or row['median_hh_income'] < 35000:
        ax.annotate(row['geo_place'], (row['median_hh_income'], row['NO2']),
                    fontsize=7.5, alpha=0.85, xytext=(5, 4), textcoords='offset points')

ax.set_xlabel('Median Household Income ($)', fontsize=12)
ax.set_ylabel('Mean Annual NO₂ (ppb)', fontsize=12)
ax.set_title('Income vs. NO₂ Pollution by NYC Neighbourhood\n'
             '(Bubble size = % Minority Population | Live ACS 2022 Data)',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3); ax.spines[['top','right']].set_visible(False)
ax.legend(fontsize=10, framealpha=0.85)
plt.tight_layout()
plt.savefig('viz10_income_no2_bubble.png')
plt.show()
print("✓ Viz 10 saved")

✓ Viz 10 saved


---
## Limitations, Ethics and Reflection

### Data Limitations

The UHF42 to census tract crosswalk is based on older boundary definitions that do not perfectly match the 2022 ACS. Pelham - Throgs Neck had to be excluded entirely and East New York required manual correction. There may be other partial mismatches we did not catch because we did not have the resources to verify all 41 neighborhoods individually.

PM2.5 is the hardest pollutant to predict in this dataset. Both tree models land at R² between 0.24 and 0.53 on the 2023 hold-out for PM2.5, compared to around 0.88 for NO2. Fine particle spikes are driven by wildfires, temperature inversions, and localized industrial events that do not appear in historical lag features. Any model trained purely on past patterns will struggle here.

We used label encoding to convert neighborhood names into integers, which technically implies an ordinal relationship that does not exist. Neighborhood coded as 30 is not more than neighborhood coded as 15. One-hot encoding or target encoding would be more appropriate but would significantly expand the feature space.

The models treat each neighborhood as completely independent. In reality pollution disperses across geographic boundaries. A high-traffic corridor affects the areas around it. We did not account for any spatial relationships.

The 2020 and 2021 years reflect the COVID period when traffic and emissions dropped sharply. Those years are included in training with no special flag, which may mean the model learned some patterns from an anomalous period and treats them as normal behavior.

### Ethical Considerations

Combining demographic data with pollution in a clustering analysis is a deliberate choice. The goal is to make the connection between race, income, and pollution exposure visible and measurable. The risk is that this type of analysis gets used to characterize communities rather than to hold the policies and decisions that created these patterns accountable. That is not our intent.

The ACS 5-year estimates average over five years of survey responses. In rapidly gentrifying areas like Williamsburg, the 2022 demographic profile may not reflect who is actually living there and breathing the air today.

These models should not be used to make resource allocation or policy decisions without expert validation. Overconfident predictions in underserved areas could be used to argue no intervention is needed when the opposite is true.

The Census API key in this notebook was exposed during development and should be considered compromised. It should be regenerated before this code is shared publicly.

---
## Summary

### Model Results

| Model | MAE | RMSE | R² | TS-CV R² |
|-------|-----|------|----|----------|
| Linear Regression | ~2.87 | ~3.87 | ~0.81 | ~-0.04 ± 0.54 |
| Random Forest | ~1.02 | ~1.35 | ~0.98 | ~0.96 ± 0.01 |
| Gradient Boosting | ~1.11 | ~1.39 | ~0.98 | ~0.96 ± 0.02 |

MAE is the average prediction error in original units: roughly 1 ppb for NO2 and 1 µg/m³ for PM2.5, both small relative to the range of values in the dataset. RMSE staying close to MAE tells us the models are not making occasional catastrophic predictions, just small consistent ones. R² of 0.98 means the models explain 98% of the variance in pollution across neighborhoods and seasons. The TimeSeriesSplit cross-validation scores matter more than the single hold-out scores because they average performance across five temporally ordered folds rather than depending on one lucky split.

The linear regression result is the most revealing. The single hold-out R² of 0.81 looked fine, but the TS-CV score collapsed to nearly zero with massive variance. That is what temporal leakage looks like. Random splitting allowed the model to train on future observations, producing the appearance of generalization when none existed. RF and GBR both held up under proper evaluation, which means they are learning real patterns.

Between RF and GBR the gap is small. RF has a slight edge on PM2.5, GBR on Ozone. Both are weakest on PM2.5 where R² on the hold-out drops to between 0.24 and 0.53. Fine particle spikes come from events like wildfires that do not show up in lag features. That is a ceiling that cannot be overcome without external data sources like weather or satellite fire detection.

### Environmental Justice

K-Means with k=4 (silhouette score 0.33) found four neighborhood groups with clearly different profiles. The most important cluster for our research question is concentrated in the Bronx. Neighborhoods like Fordham, Crotona, and Hunts Point are majority Hispanic, have median household incomes around $25,000 to $30,000, poverty rates above 40%, and elevated NO2. This is what compounded disadvantage looks like: the communities with the least capacity to respond to pollution are the ones most exposed to it.

The counterintuitive finding is that some of the wealthiest Manhattan neighborhoods have the highest absolute NO2 in the city, driven by traffic density. But high income provides real protection: better-sealed housing, air conditioning, access to healthcare, and the ability to limit outdoor time during high-pollution days. The pollution number and the health burden are not the same thing across the income spectrum.

What we found is consistent with decades of environmental justice research. Pollution in NYC is not randomly distributed. It follows lines of race and income that were shaped by decades of land use decisions about where to build highways, site industrial facilities, and invest in transit. That is a structural problem, not a coincidence.

---
## References and AI Transparency

### Dataset
NYC Environment and Health Data Portal. Air Quality Surveillance Data (PM2.5, NO2, Ozone). NYC Department of Health and Mental Hygiene. https://catalog.data.gov/dataset/air-quality

### Demographics
U.S. Census Bureau. American Community Survey 5-Year Estimates, 2022. Tables B03002, B19013, B17001. https://www.census.gov/programs-surveys/acs

### Geography
NYC Department of Health. United Hospital Fund (UHF42) Neighborhood Definitions. Used for crosswalk mapping between census tracts and health districts.

### Libraries
McKinney, W. (2010). Data Structures for Statistical Computing in Python. Proceedings of the 9th Python in Science Conference.

Pedregosa et al. (2011). Scikit-learn: Machine Learning in Python. JMLR 12, 2825-2830.

Hunter, J. D. (2007). Matplotlib: A 2D Graphics Environment. Computing in Science and Engineering, 9(3), 90-95.

Waskom, M. (2021). Seaborn: Statistical Data Visualization. JOSS, 6(60), 3021.

### AI Transparency
Claude (Anthropic) was used to help with code structure and debugging this project. It also assisted with building the Census API crosswalk pipeline. All analytical decisions, interpretations, and conclusions are our own.